# TrustOCT: Environment Setup & Data Pipeline Verification

This master notebook automates the environment initialization, Kaggle authentication, dataset acquisition, and validation tests for the **TrustOCT** project.

### Pipeline Overview
1. **Authenticate Kaggle**: Securely load Kaggle credentials to access datasets.
2. **Download & Prepare Datasets**: Fetch Kermany OCT2017 and NEH-UT datasets directly to Colab scratch disk.
3. **Install Dependencies**: Set up PyTorch, Albumentations, and other requirements.
4. **Clone Repository**: Fetch the TrustOCT modular package.
5. **Verification**: Execute `verify.py` and `statistics.py` to confirm structure and metrics.
6. **Unit Tests**: Run pytest suite to confirm dataloader functionality.

## Step 1: Connect Google Drive (Optional for saving checkpoints)

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## Step 2: Authenticate Kaggle API
Upload your `kaggle.json` credentials downloaded from Kaggle Account Settings.

In [ ]:
import os
from google.colab import files

# Upload kaggle.json file
if not os.path.exists('/root/.kaggle/kaggle.json'):
    print("Please upload your kaggle.json file:")
    uploaded = files.upload()
    !mkdir -p ~/.kaggle
    !cp kaggle.json ~/.kaggle/
    !chmod 600 ~/.kaggle/kaggle.json
    print("Kaggle API authenticated successfully.")
else:
    print("Kaggle credentials already present.")

## Step 3: Clone TrustOCT Repository
Clone the framework code from GitHub. (Update the repository URL if needed).

In [ ]:
# Define your GitHub repository details
REPO_URL = "https://github.com/Gnanapravallika/Trustworthy-OCT-AI.git"

if not os.path.exists('/content/Trustworthy-OCT-AI'):
    !git clone {REPO_URL}

%cd /content/Trustworthy-OCT-AI
!git pull

## Step 4: Install Dependencies

In [ ]:
!pip install -r requirements.txt pytest

## Step 5: Download and Prepare Datasets
We acquire the raw data directly into the folder structure expected by our configuration registry.

In [ ]:
import os

# Create local raw directories mapped in dataset.yaml
os.makedirs("datasets/raw/Kermany", exist_ok=True)
os.makedirs("datasets/raw/NEH_UT", exist_ok=True)

# 1. Download Kermany OCT2017 Dataset
if not os.path.exists("datasets/raw/Kermany/train"):
    print("Downloading Kermany OCT2017 dataset...")
    !kaggle datasets download -d paultimothymooney/kermany2018
    !unzip -q kermany2018.zip -d datasets/raw/Kermany_temp
    # Move to match configs expectations
    !mv datasets/raw/Kermany_temp/OCT2017\ /train datasets/raw/Kermany/train
    !mv datasets/raw/Kermany_temp/OCT2017\ /val datasets/raw/Kermany/val
    !mv datasets/raw/Kermany_temp/OCT2017\ /test datasets/raw/Kermany/test
    # Clean up
    !rm -rf datasets/raw/Kermany_temp kermany2018.zip
    print("Kermany dataset prepared.")
else:
    print("Kermany dataset already exists.")

# 2. Download NEH-UT Dataset
if not os.path.exists("datasets/raw/NEH_UT/train") and not os.path.exists("datasets/raw/NEH_UT/NEH-UT"):
    print("Downloading NEH-UT dataset...")
    !kaggle datasets download -d orvile/neh-ut-oct-dataset
    !unzip -q neh-ut-oct-dataset.zip -d datasets/raw/NEH_UT
    !rm neh-ut-oct-dataset.zip
    print("NEH-UT dataset prepared.")
else:
    print("NEH-UT dataset already exists.")

## Step 6: Execute Dataset Verification & Statistics
Verify the file integrity and calculate the RGB dataset normalization statistics.

In [ ]:
# Execute verification script
!python src/datasets/verify.py

# Execute stats generation script
!python src/datasets/statistics.py

## Step 7: Run Unit Test Suite
Confirm that custom transforms, bilinear filtering, and loader batch sizes work correctly on Colab.

In [ ]:
!python -m pytest tests/test_pipeline.py